In [1]:
from typing import Iterable
import time
import requests
from pathlib import Path

import numpy as np
import pandas as pd

import mygene
from brainimagelibrary import retrieve, query

In [2]:
df_path = Path('./data/gene_datasets - Gene × Dataset Matrix.csv')

In [3]:
df = pd.read_csv(df_path, index_col=0)

In [4]:
# change df to boolean
df = ~df.isna()
df.head()

,ace-ban-oat,ace-can-ill,ace-can-imp,ace-can-ink,ace-dry-dip,ace-dry-dog,ace-dry-dry,ace-dry-dub,ace-dud-vex,ace-dud-vow,...,ace-low-cap,ace-low-car,ace-low-cat,ace-low-cop,ace-low-cot,ace-low-cow,ace-low-cry,ace-low-cub,ace-low-zip,ace-met-war
Gene,,,,,,,,,,,,,,,,,,,,,
1500015O10Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1700011I03Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2900052N01Rik,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
4930509J09Rik,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
9030622O22Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Get species taxids

In [5]:
# get species names from the metadata for each dataset

species_list = []
for i in range(len(df.columns)):
    metadata = retrieve.by_id(bildid=df.columns[i])
    species = metadata['retjson'][0]['Specimen'][0]['species']
    species_list.append(species)

species_list_unique = list(set(species_list))
species_list_unique 

['human', 'mouse', 'Rhesus Macaque', 'Mouse', 'Human']

In [6]:
# count number of datasets per species
species_counts = {}
for species in species_list:
    if species in species_counts:
        species_counts[species] += 1
    else:
        species_counts[species] = 1

species_counts

{'mouse': 1, 'human': 10, 'Human': 5, 'Rhesus Macaque': 5, 'Mouse': 60}

In [7]:
def common_name_to_taxid(name):
    """Returns (taxid, scientific_name), or (None, None) if not found."""
    # Step 1: search by name to get taxid
    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params={"db": "taxonomy", "term": name, "retmode": "json"},
    )
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    if not ids:
        return None
    return int(ids[0])

def taxids_to_names(taxids):
    taxids = list(set(str(t) for t in taxids))  # dedupe
    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
        params={"db": "taxonomy", "id": ",".join(taxids), "retmode": "json"},
    )
    result = r.json().get("result", {})
    return {int(tid): result.get(tid, {}).get("scientificname") for tid in taxids}

In [8]:
# from metadata entry to taxid
taxid_map = {}
for common_name in species_list_unique:
    taxid = common_name_to_taxid(common_name)

    taxid_map[common_name] = taxid

    time.sleep(0.33)
    
# from taxid to scientific name
name_map = taxids_to_names(list(taxid_map.values()))

for common_name, taxid in taxid_map.items():
    print(f"{common_name} -> {taxid} -> {name_map.get(taxid)}")

human -> 9606 -> Homo sapiens
mouse -> 10090 -> Mus musculus
Rhesus Macaque -> 9544 -> Macaca mulatta
Mouse -> 10090 -> Mus musculus
Human -> 9606 -> Homo sapiens


## Resolve gene names

In [9]:
# find taxids for each column
def resolve(col):
    metadata = retrieve.by_id(bildid=col)
    species = metadata['retjson'][0]['Specimen'][0]['species']
    taxid = taxid_map.get(species)
    return taxid

col_taxid = {col: resolve(col) for col in df.columns}

In [10]:
# get genes for each taxid
taxid_to_genes: dict[int, set[str]] = {}
for col, taxid in col_taxid.items():
    present = df.index[df[col]]
    taxid_to_genes.setdefault(taxid, set()).update(present)

In [11]:
len(taxid_to_genes)

3

In [12]:
FIELDS = "symbol,name,taxid,entrezgene,ensembl.gene,alias,homologene"
DF_COLUMNS = ["species", "gene_query", "ensembl", "entrez", "aliases", "homolog_id"]


def is_mergeable(hits: list[dict]) -> bool:
    """Multiple mygene hits that actually describe the same biological gene.

    Covers two patterns:
      1. Pure soft duplicate — same symbol/name, each hit carries only
         ensembl XOR entrezgene (CASC6, LINC02822, etc.)
      2. Multi-Ensembl gene — same symbol/name, different Ensembl IDs across
         hits (TRBC1, IGHA1 — alt-haplotype copies in immunoreceptor loci)

    Discriminator: all hits agree on symbol, name, and taxid. If they do,
    they're the same gene and can be merged. If they don't, it's real biological ambiguity.
    """
    if len(hits) < 2:
        return False
    identities = {(h.get("symbol"), h.get("name"), h.get("taxid")) for h in hits}
    return len(identities) == 1


def _build_row(query: str, hits: list[dict], species_label: str) -> dict:
    """Flatten one or more mergeable hits into a single DataFrame row."""
    ensembl_ids, entrez_ids, aliases, homolog_ids = [], [], [], []

    for h in hits:
        ens = h.get("ensembl")
        if isinstance(ens, dict) and ens.get("gene"):
            ensembl_ids.append(ens["gene"])
        elif isinstance(ens, list):
            ensembl_ids.extend(e["gene"] for e in ens if e.get("gene"))

        if h.get("entrezgene"):
            entrez_ids.append(str(h["entrezgene"]))

        ali = h.get("alias", [])
        aliases.extend([ali] if isinstance(ali, str) else ali)

        hg = h.get("homologene")
        if isinstance(hg, dict) and hg.get("id") is not None:
            homolog_ids.append(hg["id"])

    return {
        "species": species_label,
        "gene_query": query,
        "ensembl": sorted(set(ensembl_ids)),
        "entrez": sorted(set(entrez_ids)),
        "aliases": sorted(set(aliases)),
        "homolog_id": homolog_ids[0] if homolog_ids else None,
    }


def _group_hits_by_query(out: list[dict]) -> dict[str, list[dict]]:
    grouped: dict[str, list[dict]] = {}
    for h in out:
        if h.get("notfound"):
            continue
        grouped.setdefault(h["query"], []).append(h)
    return grouped


def resolve_genes_for_species(
    genes: Iterable[str],
    taxid: int,
    species_label: str,
    mg: mygene.MyGeneInfo,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Resolve a list of gene queries against mygene; return a DataFrame with
    one row per uniquely resolved gene.

    Two-pass: strict `symbol` scope first, then alias/xref scopes for misses.
    For each query:
      • 1 hit                        → add as a row
      • >1 hits, all same identity   → merge into one row
      • >1 hits, different identities → ambiguous, print, skip
      • 0 hits in either pass        → unresolved, print, skip
    """
    genes = sorted(set(genes))
    if not genes:
        return pd.DataFrame(columns=DF_COLUMNS)

    rows: list[dict] = []
    ambiguous: list[tuple[str, list[dict]]] = []

    def handle(query: str, hits: list[dict]) -> None:
        if len(hits) == 1 or is_mergeable(hits):
            rows.append(_build_row(query, hits, species_label))
        else:
            ambiguous.append((query, hits))

    # Pass 1 — strict symbol
    r1 = mg.querymany(
        genes, 
        scopes="symbol", 
        fields=FIELDS,
        species=taxid, 
        returnall=True,
    )
    for query, hits in _group_hits_by_query(r1["out"]).items():
        handle(query, hits)
    misses = list(r1["missing"])

    # Pass 2 — alias / xref scopes on misses
    unresolved: list[str] = []
    if misses:
        r2 = mg.querymany(
            misses, scopes="alias,ensembl.gene,entrezgene,name", fields=FIELDS,
            species=taxid, returnall=True,
        )
        for query, hits in _group_hits_by_query(r2["out"]).items():
            handle(query, hits)
        unresolved = list(r2["missing"])

    if verbose:
        if ambiguous:
            print(f"[{species_label}] {len(ambiguous)} ambiguous (skipped):")
            for q, hits in ambiguous:
                syms = sorted({h.get("symbol", "?") for h in hits})
                print(f"  {q} → {syms}")
        if unresolved:
            print(f"[{species_label}] {len(unresolved)} unresolved (no match): {unresolved}")

    return pd.DataFrame(rows, columns=DF_COLUMNS)

In [13]:
mg = mygene.MyGeneInfo()

df_list = []
for taxid in taxid_to_genes:
    species_name = name_map.get(taxid)
    print(f"Resolving genes for {species_name} (taxid {taxid})...")
    genes = taxid_to_genes[taxid]
    temp = resolve_genes_for_species(genes, taxid=taxid, species_label=species_name, mg=mg)
    
    print(f'Resolved {len(temp)} / {len(genes)} genes for {species_name} (taxid {taxid})')
    print(f'Number of genes without homologene IDs: {(temp["homolog_id"].isna()).sum()} / {len(temp)}\n')

    df_list.append(temp)

genes_resolved = pd.concat(df_list, ignore_index=True)

Resolving genes for Mus musculus (taxid 10090)...


1 input query terms found dup hits:	[('B230209E15Rik', 2)]
48 input query terms found no hit:	['1500015O10Rik', '1700011I03Rik', 'Cars', 'Fyb', 'Hist1h1d', 'Lhfp', 'Prrxl1', 'Sept4', 'blank-0', 
2 input query terms found dup hits:	[('Lhfp', 2), ('volume', 10)]
38 input query terms found no hit:	['blank-0', 'blank-1', 'blank-10', 'blank-11', 'blank-12', 'blank-13', 'blank-14', 'blank-15', 'blan


[Mus musculus] 2 ambiguous (skipped):
  Lhfp → ['Lhfpl1', 'Lhfpl6']
  volume → ['Blacv1', 'Brnvl1', 'Brnvl2', 'Dshv1', 'Dshv2', 'Grv', 'Mcvq1', 'Mcvq2', 'Mcvq3', 'Mpvq2']
[Mus musculus] 38 unresolved (no match): ['blank-0', 'blank-1', 'blank-10', 'blank-11', 'blank-12', 'blank-13', 'blank-14', 'blank-15', 'blank-16', 'blank-17', 'blank-18', 'blank-19', 'blank-2', 'blank-20', 'blank-21', 'blank-22', 'blank-23', 'blank-24', 'blank-25', 'blank-26', 'blank-3', 'blank-4', 'blank-5', 'blank-6', 'blank-7', 'blank-8', 'blank-9', 'blank_IEG-0', 'blank_IEG-1', 'blank_IEG-2', 'blank_IEG-3', 'fov_x', 'fov_y', 'global_x', 'global_y', 'mRNA_counts', 'median_total_density', 'polyT']
Resolved 1144 / 1184 genes for Mus musculus (taxid 10090)
Number of genes without homologene IDs: 30 / 1144

Resolving genes for Homo sapiens (taxid 9606)...


10 input query terms found dup hits:	[('CASC6', 2), ('DLX6-AS1', 2), ('IGHA1', 2), ('LINC01965', 2), ('LINC02822', 2), ('NR2F2-AS1', 2), 
5 input query terms found no hit:	['FAM155A', 'FAM160A1', 'FAM189A2', 'NDUFA4L2', 'TCTEX1D1']


Resolved 1005 / 1005 genes for Homo sapiens (taxid 9606)
Number of genes without homologene IDs: 38 / 1005

Resolving genes for Macaca mulatta (taxid 9544)...


4 input query terms found dup hits:	[('ADCYAP1', 2), ('CLDN5', 2), ('DPF3', 2), ('TSHZ2', 2)]
1 input query terms found no hit:	['ZNF429']


Resolved 300 / 300 genes for Macaca mulatta (taxid 9544)
Number of genes without homologene IDs: 42 / 300



In [14]:
# save
genes_resolved.to_pickle('./data/genes_resolved.pkl')

In [15]:
len(genes_resolved)

2449

From Ensembl documentation: human and mouse have multiple paths through the genome, including haplotypes — different versions of particular regions found in different individuals — which are equally valid compared to the primary assembly. 

The convention is that each assembly path gets its own ENSG ID, but NCBI cross-references only the primary one. So we end up with N Ensembl records per haplotype/patch, all pointing biologically to the same gene, all sharing one Entrez ID. 

In [16]:
genes_resolved.loc[genes_resolved.ensembl.apply(lambda x: len(x) > 1),:]

,species,gene_query,ensembl,entrez,aliases,homolog_id
1165,Homo sapiens,AKT3,"[ENSG00000117020, ENSG00000275199]",[10000],"[MPPH, MPPH2, PKB-GAMMA, PKBG, PRKBG, RAC-PK-g...",55904.0
1204,Homo sapiens,BLK,"[ENSG00000136573, ENSG00000285369]",[640],[MODY11],1297.0
1226,Homo sapiens,CALB2,"[ENSG00000172137, ENSG00000282830]",[794],"[CAB29, CAL2, CR]",1318.0
1246,Homo sapiens,CCL3,"[ENSG00000274221, ENSG00000277632, ENSG0000027...",[6348],"[G0S19-1, LD78, LD78ALPHA, MIP-1-alpha, MIP1A,...",88430.0
1247,Homo sapiens,CCL4,"[ENSG00000275302, ENSG00000275824, ENSG0000027...",[6351],"[ACT2, AT744.1, G-26, HC21, LAG-1, LAG1, MIP-1...",48153.0
1248,Homo sapiens,CCL5,"[ENSG00000271503, ENSG00000274233]",[6352],"[D17S136E, RANTES, SCYA5, SIS-delta, SISd, TCP...",2244.0
1302,Homo sapiens,CD99,"[ENSG00000002586, ENSG00000292348]",[4267],"[HBA71, MIC2, MIC2X, MIC2Y, MSK5X]",48107.0
1344,Homo sapiens,CNTNAP2,"[ENSG00000174469, ENSG00000278728]",[26047],"[AUTS15, CASPR2, CDFE, NRXN4, PTHSL1]",69159.0
1372,Homo sapiens,CSF2RA,"[ENSG00000198223, ENSG00000292357]",[1438],"[CD116, CDw116, CSF2R, CSF2RAX, CSF2RAY, CSF2R...",48406.0
1408,Homo sapiens,DERL3,"[ENSG00000099958, ENSG00000274437]",[91319],"[C22orf14, IZP6, LLN2, derlin-3]",41566.0


## Build entrez to datasets data frame

In [17]:
from typing import Callable
import pandas as pd


def build_presence_by_entrez(
    original_presence: pd.DataFrame,
    resolved: pd.DataFrame,
    get_species_label: Callable[[str], str],
    verbose: bool = True,
) -> pd.DataFrame:
    """Re-key BIL's presence matrix from raw symbol strings to canonical
    Entrez IDs, respecting species boundaries.

    Parameters
    ----------
    original_presence : index = gene_query (raw BIL symbols), columns = dataset_id,
        values = bool. 
    resolved : output of resolve_genes_for_species() concatenated across species.
        Required columns: species, gene_query, entrez.
    get_species_label : dataset_id -> species_label (the same label used in
        resolved["species"], e.g. "human"). Compose your existing get_taxid()
        with a {taxid: label} map.
    """
    # 1. Map each dataset column to its species label
    ds_species = pd.Series({c: get_species_label(c) for c in original_presence.columns})

    # 2. Clean the presence matrix to plain booleans
    pres = original_presence.fillna(False).astype(bool)

    # 3. Pull the canonical entrez (scalar) out of each resolved row's list
    resolved = resolved.copy()
    resolved["_entrez"] = resolved["entrez"].apply(
        lambda xs: xs[0] if isinstance(xs, list) and xs else None
    )
    n_no_entrez = int(resolved["_entrez"].isna().sum())
    resolved = resolved.dropna(subset=["_entrez"])

    # 4. For each (gene_query, species), pull its presence row and mask to
    #    dataset columns of the matching species
    rows: list[pd.Series] = []
    n_missing = 0
    for _, r in resolved.iterrows():
        gq = r["gene_query"]
        if gq not in pres.index:
            n_missing += 1
            continue
        row = pres.loc[gq]
        if isinstance(row, pd.DataFrame):           # duplicate index in original
            row = row.any(axis=0)
        masked = row & (ds_species == r["species"])
        masked.name = r["_entrez"]
        rows.append(masked)

    # 5. Stack; OR across any duplicate entrez (multiple gene_query -> same gene)
    if not rows:
        return pd.DataFrame(columns=original_presence.columns).astype(bool)
    out = pd.concat(rows, axis=1).T
    out = out.groupby(level=0).any().astype(bool)
    out.index.name = "entrez"

    if verbose:
        print(f"resolved input rows:                {len(resolved) + n_no_entrez}")
        print(f"  skipped — no entrez:              {n_no_entrez}")
        print(f"  skipped — gene_query not in presence: {n_missing}")
        print(f"output: {len(out)} genes × {len(out.columns)} datasets")

    return out

In [18]:
# callable from dataset name to scientif species name
def common_to_scientific(dataset_id: str) -> str:
    metadata = retrieve.by_id(bildid=dataset_id)
    common_name = metadata['retjson'][0]['Specimen'][0]['species']
    taxid = taxid_map.get(common_name)
    scientific_name = name_map.get(taxid)
    return scientific_name

presence_by_entrez = build_presence_by_entrez(
    df,
    genes_resolved,
    get_species_label=common_to_scientific,
)

resolved input rows:                2449
  skipped — no entrez:              0
  skipped — gene_query not in presence: 0
output: 2448 genes × 81 datasets


In [19]:
presence_by_entrez.head()

,ace-ban-oat,ace-can-ill,ace-can-imp,ace-can-ink,ace-dry-dip,ace-dry-dog,ace-dry-dry,ace-dry-dub,ace-dud-vex,ace-dud-vow,...,ace-low-cap,ace-low-car,ace-low-cat,ace-low-cop,ace-low-cot,ace-low-cow,ace-low-cry,ace-low-cub,ace-low-zip,ace-met-war
entrez,,,,,,,,,,,,,,,,,,,,,
10000,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
100038489,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
100039691,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
100039795,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
100042150,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [20]:
presence_by_entrez.to_pickle('./data/presence_by_entrez.pkl')

In [21]:
print("Total True cells:", presence_by_entrez.sum().sum())
print("Genes with ≥1 dataset:", (presence_by_entrez.sum(axis=1) > 0).sum())

Total True cells: 36230
Genes with ≥1 dataset: 2448


## Build a datasets info

In [23]:
datasets = pd.DataFrame(columns=["title", "species", "url"], index=pd.Index(df.columns, name="dataset_id"))

for col in df.columns:
    metadata = retrieve.by_id(bildid=col)
    common_name = metadata['retjson'][0]['Specimen'][0]['species']
    taxid = taxid_map.get(common_name)
    scientific_name = name_map.get(taxid)
    url = metadata['retjson'][0]['Dataset'][0]['bildirectory']
    title = metadata['retjson'][0]['Dataset'][0]['title']

    datasets.loc[col] = {
        "title": title,
        "species": scientific_name,
        "url": url,
    }


In [24]:
datasets.species.value_counts()

species
Mus musculus      61
Homo sapiens      15
Macaca mulatta     5
Name: count, dtype: int64

In [25]:
datasets.to_pickle('./data/datasets.pkl')